In [20]:
%pip install yfinance -q

Note: you may need to restart the kernel to use updated packages.


# 📈 Stock Forecast Project — Step 1 (v3): Data Sources + EDA + External Finance APIs + Excel Export

**Date:** 2026-01-15  
**Scope:** EDA + feature decisions + data ingestion connectors. No model training yet.

---

## What changed in v3
✅ Added connectors for free financial APIs:
- **Alpha Vantage** (fundamentals / overview, limited rate)  
- **Finnhub** (company profile / basic financials / analyst targets, limited rate)  
- **Financial Modeling Prep (FMP)** (profiles / ratios / key metrics / quote, limited rate)

✅ Added a final step to **export all EDA outputs to an Excel workbook** (multi-sheet) for visual review.

---

## Security note (important)
Do **not** hardcode API keys into notebooks or commit them to Git.  
This notebook reads keys from environment variables (or an optional `.env` file).


## 0) API Keys Setup (Environment Variables)

Set these variables in your environment:

- `ALPHA_VANTAGE_API_KEY`
- `FINNHUB_API_KEY`
- `FMP_API_KEY`

### Option A — Temporary in notebook (OK for local, don’t commit)
You can set:

```python
import os
os.environ["ALPHA_VANTAGE_API_KEY"] = "..."
os.environ["FINNHUB_API_KEY"] = "..."
os.environ["FMP_API_KEY"] = "..."
```

### Option B — `.env` file (recommended)
Create a `.env` file next to the notebook:

```
ALPHA_VANTAGE_API_KEY=...
FINNHUB_API_KEY=...
FMP_API_KEY=...
```

and the notebook can load it if `python-dotenv` is installed.


In [19]:
# Core
import os
from pathlib import Path
from datetime import datetime
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Provider: Yahoo Finance for OHLCV
import yfinance as yf

# HTTP for APIs
import requests

# Feature ranking
from sklearn.feature_selection import mutual_info_classif

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 180)


ModuleNotFoundError: No module named 'yfinance'

In [ ]:
# Optional: load .env if available
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded .env (if present).")
except Exception:
    print("python-dotenv not installed (optional). You can still use OS environment variables.")


Loaded .env (if present).


In [ ]:
# ==========================
# API Keys (fill in locally)
# ==========================
# ✅ Fill these strings if you want to run the notebook without environment variables.
# ⚠️ Do NOT commit your keys to Git. Keep this notebook local or use environment variables / .env.

# Initialize as empty (will be read from os.getenv/.env below)
ALPHA_VANTAGE_API_KEY = ""
FINNHUB_API_KEY = ""
FMP_API_KEY = ""

# Optional: also push them into environment variables for downstream code.
# (Leaving them empty means we'll fall back to os.getenv / .env)
if ALPHA_VANTAGE_API_KEY:
    os.environ["ALPHA_VANTAGE_API_KEY"] = ALPHA_VANTAGE_API_KEY
if FINNHUB_API_KEY:
    os.environ["FINNHUB_API_KEY"] = FINNHUB_API_KEY
if FMP_API_KEY:
    os.environ["FMP_API_KEY"] = FMP_API_KEY


## 1) Configuration

### Practical note about API limits
Fetching fundamentals for **500 tickers** from free tiers can hit rate limits quickly.

This notebook therefore has:
- **caching** of raw responses and final tables
- a parameter `FUNDAMENTALS_MAX_TICKERS` (you can increase gradually)
- optional provider priority: FMP → Finnhub → Alpha Vantage (fallback)

For EDA, start with 50–150 tickers for fundamentals, and keep OHLCV for all.


In [ ]:
OUTDIR = Path("out_eda")
OUTDIR.mkdir(exist_ok=True, parents=True)

CACHE_DIR = OUTDIR / "cache"
CACHE_DIR.mkdir(exist_ok=True, parents=True)

# Universe
TICKERS_FILE = None  # e.g. "universe.txt"
TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "TSLA", "JPM", "XOM", "SPY"]

# OHLCV source
START_DATE = "2018-01-01"
END_DATE = None
INTERVAL = "1d"
AUTO_ADJUST = True

# Weekly horizon (5 trading days)
HORIZON_DAYS = 5

# Download chunk size (yfinance)
CHUNK_SIZE = 100
MIN_ROWS_PER_TICKER = 252

# Market features
MARKET_TICKER = "SPY"
ADD_MARKET_FEATURES = True

# External fundamentals snapshot (APIs)
ADD_EXTERNAL_FUNDAMENTALS = True
FUNDAMENTALS_MAX_TICKERS = 150  # increase gradually (API limits)
PROVIDER_PRIORITY = ["fmp", "finnhub", "alphavantage"]  # fallback order

# Throttling / retry
HTTP_TIMEOUT = 30
MAX_RETRIES = 3

# Rate-limits (tune based on your plan)
# Alpha Vantage free: typically 5 calls/min. Keep low.
SLEEP_ALPHA = 12.5
# Finnhub / FMP free tiers: vary; keep small to be safe.
SLEEP_FINNHUB = 0.35
SLEEP_FMP = 0.35

# EDA speed controls
SAMPLE_ROWS_FOR_CORR = 50_000
RANDOM_SEED = 42

SAVE_FULL_DATASET = False  # parquet can be large

# Excel export
EXCEL_PATH = OUTDIR / "EDA_Export.xlsx"
MAX_ROWS_PER_SHEET = 200_000  # avoid huge Excel


## 2) Load Universe (Tickers)

In [ ]:
def read_tickers(tickers, tickers_file):
    if tickers_file:
        p = Path(tickers_file)
        lines = [ln.strip().upper() for ln in p.read_text(encoding="utf-8").splitlines()]
        lines = [ln for ln in lines if ln and not ln.startswith("#")]
        seen, out = set(), []
        for t in lines:
            if t not in seen:
                out.append(t)
                seen.add(t)
        return out
    seen, out = set(), []
    for t in tickers:
        t = str(t).strip().upper()
        if t and t not in seen:
            out.append(t)
            seen.add(t)
    return out

tickers = read_tickers(TICKERS, TICKERS_FILE)

# Ensure market ticker exists if needed
if ADD_MARKET_FEATURES and MARKET_TICKER not in tickers:
    tickers = tickers + [MARKET_TICKER]

print(f"Tickers loaded: {len(tickers)}")
tickers[:20]


Tickers loaded: 10


['AAPL', 'MSFT', 'NVDA', 'AMZN', 'META', 'GOOGL', 'TSLA', 'JPM', 'XOM', 'SPY']

## 3) Download OHLCV (Yahoo via yfinance)

In [ ]:
def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def download_yahoo_ohlcv(tickers, start, end, interval="1d", auto_adjust=True, chunk_size=100):
    end = end or datetime.utcnow().strftime("%Y-%m-%d")
    frames = []
    for ch in chunked(tickers, chunk_size):
        df = yf.download(
            tickers=" ".join(ch),
            start=start,
            end=end,
            interval=interval,
            auto_adjust=auto_adjust,
            group_by="ticker",
            threads=True,
            progress=False,
        )
        frames.append(df)
    if len(frames) == 1:
        return frames[0]
    return pd.concat(frames, axis=1)

raw = download_yahoo_ohlcv(
    tickers=tickers,
    start=START_DATE,
    end=END_DATE,
    interval=INTERVAL,
    auto_adjust=AUTO_ADJUST,
    chunk_size=CHUNK_SIZE,
)

raw.shape


C:\Users\Yotam\AppData\Local\Temp\ipykernel_27744\1302740277.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = end or datetime.utcnow().strftime("%Y-%m-%d")


NameError: name 'yf' is not defined

## 4) Standardize OHLCV to long panel

In [ ]:
def standardize_ohlcv(raw: pd.DataFrame) -> pd.DataFrame:
    if raw.empty:
        raise ValueError("No data downloaded.")
    df = raw.copy()
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"

    if not isinstance(df.columns, pd.MultiIndex):
        cols_map = {c.lower(): c for c in df.columns}
        needed = ["open", "high", "low", "close", "volume"]
        if any(c not in cols_map for c in needed):
            raise ValueError(f"Missing required columns. Got: {list(df.columns)}")
        out = df.rename(columns={
            cols_map["open"]: "open",
            cols_map["high"]: "high",
            cols_map["low"]: "low",
            cols_map["close"]: "close",
            cols_map["volume"]: "volume",
        }).reset_index()
        out["ticker"] = "SINGLE"
        return out[["date","ticker","open","high","low","close","volume"]]

    # Handle yfinance v1.0+ format: columns like (ticker, field)
    # Check which level contains field names and which contains tickers
    lvl0 = set(map(str.lower, df.columns.get_level_values(0).unique()))
    fields = {"open","high","low","close","volume","adj close"}
    
    # yfinance v1.0+ puts tickers at level 0, fields at level 1
    # But handle both old and new formats
    if any(f in lvl0 for f in fields):
        field_level = 0
        ticker_level = 1
    else:
        field_level = 1
        ticker_level = 0

    out_frames = []
    tickers2 = df.columns.get_level_values(ticker_level).unique()
    for t in tickers2:
        try:
            sub = df.xs(t, axis=1, level=ticker_level, drop_level=False)
            sub_cols = sub.columns.get_level_values(field_level)
            sub.columns = [str(c).lower() for c in sub_cols]
            if "close" not in sub.columns and "adj close" in sub.columns:
                sub = sub.rename(columns={"adj close": "close"})
            needed = ["open","high","low","close","volume"]
            if any(c not in sub.columns for c in needed):
                continue
            tmp = sub[needed].copy()
            tmp["ticker"] = str(t)
            out_frames.append(tmp)
        except Exception:
            continue

    if not out_frames:
        raise ValueError(f"Could not parse any tickers from columns: {list(df.columns)}")
    
    out = pd.concat(out_frames, axis=0).reset_index()
    out = out.rename(columns={"index":"date"}) if "index" in out.columns else out
    out["date"] = pd.to_datetime(out["date"])
    out = out.sort_values(["ticker","date"]).reset_index(drop=True)
    return out[["date","ticker","open","high","low","close","volume"]]

panel = standardize_ohlcv(raw)
panel.head()


## 5) Coverage & Missingness (raw overview)

In [ ]:
def raw_overview(panel: pd.DataFrame) -> pd.DataFrame:
    g = panel.groupby("ticker")
    out = pd.DataFrame({
        "rows": g.size(),
        "start": g["date"].min(),
        "end": g["date"].max(),
        "missing_open": g["open"].apply(lambda s: s.isna().mean()),
        "missing_close": g["close"].apply(lambda s: s.isna().mean()),
        "missing_volume": g["volume"].apply(lambda s: s.isna().mean()),
        "mean_volume": g["volume"].mean(),
        "median_close": g["close"].median(),
    }).reset_index()
    return out.sort_values("rows", ascending=False)

overview = raw_overview(panel)
overview.to_csv(OUTDIR / "raw_overview.csv", index=False, encoding="utf-8")
overview.head(12)


### Filter tickers with enough history

In [ ]:
good = overview[overview["rows"] >= MIN_ROWS_PER_TICKER]["ticker"].tolist()
print(f"Tickers passing MIN_ROWS_PER_TICKER={MIN_ROWS_PER_TICKER}: {len(good)} / {len(overview)}")
panel_f = panel[panel["ticker"].isin(good)].copy()
panel_f.shape, panel_f["ticker"].nunique()


## 6) EDA Helpers (plots)

In [ ]:
def plot_hist(series: pd.Series, title: str, path: Path, bins: int = 60):
    plt.figure()
    s = series.dropna()
    plt.hist(s.values, bins=bins)
    plt.title(title)
    plt.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=140)
    plt.show()

def plot_bar(x, y, title, path, rotate=60):
    plt.figure(figsize=(10,4))
    plt.bar(x, y)
    plt.title(title)
    plt.xticks(rotation=rotate, ha="right")
    plt.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=140)
    plt.show()


## 7) Sanity checks

In [ ]:
example_ticker = panel_f["ticker"].iloc[0]
g = panel_f[panel_f["ticker"] == example_ticker].copy()
g["ret_1"] = g["close"].pct_change(1)

plot_hist(g["ret_1"], f"{example_ticker} | Daily returns distribution", OUTDIR / "figures" / "example_daily_returns.png")
plot_hist(np.log1p(g["volume"]), f"{example_ticker} | log(1+volume) distribution", OUTDIR / "figures" / "example_log_volume.png")


## 8) Features + Targets (includes MACD + Finviz-like perf)

This section creates the **trainable** features computed purely from historical OHLCV (safe for backtests):
- Returns / vol
- SMA/EMA distances
- RSI
- **MACD (asked explicitly)**
- ATR / Bollinger width
- Relative volume
- Perf week/month/quarter/… & YTD
- Optional market features (SPY return/vol + rolling beta)

Targets (weekly horizon):
- Direction, return, max-up, max-down


In [ ]:
def ema(series: pd.Series, span: int) -> pd.Series:
    return series.ewm(span=span, adjust=False).mean()

def rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / (avg_loss.replace(0, np.nan))
    return 100 - (100 / (1 + rs))

def true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr

def forward_rolling_max(s: pd.Series, window: int) -> pd.Series:
    rev = s.iloc[::-1]
    return rev.shift(1).rolling(window, min_periods=1).max().iloc[::-1]

def forward_rolling_min(s: pd.Series, window: int) -> pd.Series:
    rev = s.iloc[::-1]
    return rev.shift(1).rolling(window, min_periods=1).min().iloc[::-1]

def perf_return(close: pd.Series, days: int) -> pd.Series:
    return close / close.shift(days) - 1

def ytd_return(dates: pd.Series, close: pd.Series) -> pd.Series:
    year = dates.dt.year
    first_close = close.groupby(year).transform("first")
    return close / first_close - 1

def add_features_and_targets(panel: pd.DataFrame, horizon: int, add_market: bool = True, market_ticker: str = "SPY"):
    df = panel.copy().sort_values(["ticker","date"]).reset_index(drop=True)

    # Market series
    mkt = None
    if add_market:
        m = df[df["ticker"] == market_ticker][["date","close"]].copy().sort_values("date")
        m["mkt_ret_1"] = m["close"].pct_change(1)
        m["mkt_vol_20"] = np.log(m["close"]).diff().rolling(20).std() * np.sqrt(252)
        mkt = m[["date","mkt_ret_1","mkt_vol_20"]]

    frames = []
    for t, g in df.groupby("ticker", sort=False):
        g = g.copy()

        # Returns / log returns
        g["ret_1"] = g["close"].pct_change(1)
        g["ret_3"] = g["close"].pct_change(3)
        g["ret_5"] = g["close"].pct_change(5)
        g["ret_10"] = g["close"].pct_change(10)
        g["ret_20"] = g["close"].pct_change(20)
        g["logret_1"] = np.log(g["close"]).diff()

        # Volatility (Finviz-like)
        g["vol_5"] = g["logret_1"].rolling(5).std() * np.sqrt(252)
        g["vol_20"] = g["logret_1"].rolling(20).std() * np.sqrt(252)

        # Trend
        g["sma_10"] = g["close"].rolling(10).mean()
        g["sma_20"] = g["close"].rolling(20).mean()
        g["sma_50"] = g["close"].rolling(50).mean()
        g["sma_200"] = g["close"].rolling(200).mean()
        g["ema_12"] = ema(g["close"], 12)
        g["ema_26"] = ema(g["close"], 26)

        g["close_sma10"] = g["close"] / g["sma_10"] - 1
        g["close_sma20"] = g["close"] / g["sma_20"] - 1
        g["close_sma50"] = g["close"] / g["sma_50"] - 1
        g["close_sma200"] = g["close"] / g["sma_200"] - 1
        g["close_ema12"] = g["close"] / g["ema_12"] - 1
        g["close_ema26"] = g["close"] / g["ema_26"] - 1

        # RSI / MACD
        g["rsi_14"] = rsi(g["close"], 14)
        g["macd"] = g["ema_12"] - g["ema_26"]
        g["macd_signal"] = ema(g["macd"], 9)
        g["macd_hist"] = g["macd"] - g["macd_signal"]

        # Range / gap / volume
        g["range_pct"] = (g["high"] - g["low"]) / g["close"].replace(0, np.nan)
        g["gap_pct"] = g["open"] / g["close"].shift(1) - 1
        g["vol_chg_1"] = g["volume"].pct_change(1)
        g["avg_volume_20"] = g["volume"].rolling(20).mean()
        g["rel_volume_20"] = g["volume"] / g["avg_volume_20"]
        g["vol_z_20"] = (g["volume"] - g["avg_volume_20"]) / g["volume"].rolling(20).std()

        # ATR / Bollinger width
        tr = true_range(g["high"], g["low"], g["close"])
        g["atr_14"] = tr.rolling(14).mean()
        mid = g["close"].rolling(20).mean()
        sd = g["close"].rolling(20).std()
        upper = mid + 2 * sd
        lower = mid - 2 * sd
        g["bb_width"] = (upper - lower) / mid.replace(0, np.nan)

        # Performance (Finviz-like)
        g["perf_week"] = perf_return(g["close"], 5)
        g["perf_month"] = perf_return(g["close"], 21)
        g["perf_quarter"] = perf_return(g["close"], 63)
        g["perf_half_year"] = perf_return(g["close"], 126)
        g["perf_year"] = perf_return(g["close"], 252)
        g["perf_3y"] = perf_return(g["close"], 252*3)
        g["perf_5y"] = perf_return(g["close"], 252*5)
        g["perf_10y"] = perf_return(g["close"], 252*10)
        g["perf_ytd"] = ytd_return(g["date"], g["close"])

        # Calendar
        g["dow"] = g["date"].dt.dayofweek
        g["month"] = g["date"].dt.month

        # Targets
        g[f"target_return_{horizon}d"] = g["close"].shift(-horizon) / g["close"] - 1
        g[f"target_dir_{horizon}d"] = (g[f"target_return_{horizon}d"] > 0).astype("int8")
        high_fwd = forward_rolling_max(g["high"], horizon)
        low_fwd = forward_rolling_min(g["low"], horizon)
        g[f"target_max_up_{horizon}d"] = high_fwd / g["close"] - 1
        g[f"target_max_down_{horizon}d"] = low_fwd / g["close"] - 1

        frames.append(g)

    out = pd.concat(frames, axis=0).sort_values(["ticker","date"]).reset_index(drop=True)

    # Merge market features + beta
    if add_market and mkt is not None:
        out = out.merge(mkt, on="date", how="left")
        out["beta_252"] = np.nan
        for t, g in out.groupby("ticker", sort=False):
            if t == market_ticker:
                continue
            s = g[["ret_1","mkt_ret_1"]].copy()
            cov = s["ret_1"].rolling(252).cov(s["mkt_ret_1"])
            var = s["mkt_ret_1"].rolling(252).var()
            beta = cov / var.replace(0, np.nan)
            out.loc[g.index, "beta_252"] = beta.values

    feature_cols = [
        "ret_1","ret_3","ret_5","ret_10","ret_20","logret_1",
        "vol_5","vol_20",
        "close_sma10","close_sma20","close_sma50","close_sma200","close_ema12","close_ema26",
        "rsi_14","macd","macd_signal","macd_hist",
        "range_pct","gap_pct",
        "vol_chg_1","avg_volume_20","rel_volume_20","vol_z_20",
        "atr_14","bb_width",
        "perf_week","perf_month","perf_quarter","perf_half_year","perf_year","perf_ytd","perf_3y","perf_5y","perf_10y",
        "dow","month"
    ]
    if add_market:
        feature_cols += ["mkt_ret_1","mkt_vol_20","beta_252"]

    return out, feature_cols

dataset, trainable_feature_cols = add_features_and_targets(
    panel_f,
    horizon=HORIZON_DAYS,
    add_market=ADD_MARKET_FEATURES,
    market_ticker=MARKET_TICKER
)

target_dir_col = f"target_dir_{HORIZON_DAYS}d"
target_ret_col = f"target_return_{HORIZON_DAYS}d"
target_max_up_col = f"target_max_up_{HORIZON_DAYS}d"
target_max_dn_col = f"target_max_down_{HORIZON_DAYS}d"

dataset[["date","ticker"] + trainable_feature_cols + [target_dir_col, target_ret_col]].head()


## 9) Target distributions

In [ ]:
plot_hist(dataset[target_ret_col], f"Weekly target return ({HORIZON_DAYS}d)", OUTDIR / "figures" / "target_return_hist.png")
plot_hist(dataset[target_max_up_col], f"Weekly max-up potential ({HORIZON_DAYS}d)", OUTDIR / "figures" / "target_max_up_hist.png")
plot_hist(dataset[target_max_dn_col], f"Weekly max-down potential ({HORIZON_DAYS}d)", OUTDIR / "figures" / "target_max_down_hist.png")

print("Positive direction rate:", round(float(dataset[target_dir_col].mean()), 4))


## 10) External Fundamentals (Alpha Vantage / Finnhub / FMP)

We pull **snapshot** fundamentals for exploration and for *today's* recommendations.
For historical training/backtests, you should later add **point-in-time** fundamentals (EDGAR/paid provider with publish dates).

### What we fetch (examples)
- Market cap, enterprise value, shares outstanding, float, short metrics
- Valuation ratios (P/E, P/B, P/S, EV/EBITDA, etc.)
- Margins, ROA/ROE, debt ratios
- Analyst target / recommendation (where available)

### Caching
Each provider response is cached under `out_eda/cache/` to reduce repeated calls.


In [ ]:
ALPHA_VANTAGE_API_KEY = (ALPHA_VANTAGE_API_KEY or os.getenv("ALPHA_VANTAGE_API_KEY", "")).strip()
FINNHUB_API_KEY = (FINNHUB_API_KEY or os.getenv("FINNHUB_API_KEY", "")).strip()
FMP_API_KEY = (FMP_API_KEY or os.getenv("FMP_API_KEY", "")).strip()

print("Keys loaded:",
      "AlphaVantage" if ALPHA_VANTAGE_API_KEY else "AlphaVantage(MISSING)",
      "Finnhub" if FINNHUB_API_KEY else "Finnhub(MISSING)",
      "FMP" if FMP_API_KEY else "FMP(MISSING)")


In [ ]:
def http_get_json(url: str, params: dict, timeout: int, max_retries: int, sleep_sec: float, cache_key: str):
    cache_path = CACHE_DIR / f"{cache_key}.json"
    if cache_path.exists():
        try:
            return json.loads(cache_path.read_text(encoding="utf-8"))
        except Exception:
            pass

    last_err = None
    for i in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code == 429:
                time.sleep(sleep_sec * 3)
                continue
            r.raise_for_status()
            data = r.json()
            cache_path.write_text(json.dumps(data), encoding="utf-8")
            time.sleep(sleep_sec)
            return data
        except Exception as e:
            last_err = e
            time.sleep(sleep_sec * (i + 1))
    raise RuntimeError(f"HTTP failed after retries. url={url} err={last_err}")

def safe_num(x):
    if x is None:
        return None
    try:
        if isinstance(x, bool):
            return None
        if isinstance(x, (int, float, np.integer, np.floating)):
            if math.isfinite(float(x)):
                return float(x)
            return None
        # strings
        s = str(x).strip()
        if s in ("", "None", "nan", "NaN", "N/A", "-"):
            return None
        v = float(s)
        return v if math.isfinite(v) else None
    except Exception:
        return None


In [ ]:
# ---- FMP ----
def fetch_fmp_snapshot(ticker: str) -> dict:
    if not FMP_API_KEY:
        return {}
    base = "https://financialmodelingprep.com/api/v3"
    # profile gives market cap/beta/volAvg
    prof = http_get_json(
        url=f"{base}/profile/{ticker}",
        params={"apikey": FMP_API_KEY},
        timeout=HTTP_TIMEOUT,
        max_retries=MAX_RETRIES,
        sleep_sec=SLEEP_FMP,
        cache_key=f"fmp_profile_{ticker}",
    )
    quote = http_get_json(
        url=f"{base}/quote/{ticker}",
        params={"apikey": FMP_API_KEY},
        timeout=HTTP_TIMEOUT,
        max_retries=MAX_RETRIES,
        sleep_sec=SLEEP_FMP,
        cache_key=f"fmp_quote_{ticker}",
    )
    ratios = http_get_json(
        url=f"{base}/ratios-ttm/{ticker}",
        params={"apikey": FMP_API_KEY},
        timeout=HTTP_TIMEOUT,
        max_retries=MAX_RETRIES,
        sleep_sec=SLEEP_FMP,
        cache_key=f"fmp_ratios_ttm_{ticker}",
    )

    # These endpoints return lists; take first item
    p0 = prof[0] if isinstance(prof, list) and len(prof) else {}
    q0 = quote[0] if isinstance(quote, list) and len(quote) else {}
    r0 = ratios[0] if isinstance(ratios, list) and len(ratios) else {}

    return {
        "ticker": ticker,
        # Profile
        "market_cap": safe_num(p0.get("mktCap")),
        "beta": safe_num(p0.get("beta")),
        "avg_volume": safe_num(p0.get("volAvg")),
        "company_name": p0.get("companyName"),
        "sector": p0.get("sector"),
        "industry": p0.get("industry"),
        # Quote
        "price": safe_num(q0.get("price")),
        "prev_close": safe_num(q0.get("previousClose")),
        "changes_percentage": safe_num(str(q0.get("changesPercentage", "")).replace("%","")),
        "shares_outstanding": safe_num(q0.get("sharesOutstanding")),
        "year_high": safe_num(q0.get("yearHigh")),
        "year_low": safe_num(q0.get("yearLow")),
        # Ratios TTM (some Finviz-like)
        "pe_ttm": safe_num(r0.get("priceEarningsRatioTTM")),
        "pb_ttm": safe_num(r0.get("priceToBookRatioTTM")),
        "ps_ttm": safe_num(r0.get("priceToSalesRatioTTM")),
        "current_ratio_ttm": safe_num(r0.get("currentRatioTTM")),
        "quick_ratio_ttm": safe_num(r0.get("quickRatioTTM")),
        "debt_to_equity_ttm": safe_num(r0.get("debtEquityRatioTTM")),
        "gross_margin_ttm": safe_num(r0.get("grossProfitMarginTTM")),
        "operating_margin_ttm": safe_num(r0.get("operatingProfitMarginTTM")),
        "net_margin_ttm": safe_num(r0.get("netProfitMarginTTM")),
        "roe_ttm": safe_num(r0.get("returnOnEquityTTM")),
        "roa_ttm": safe_num(r0.get("returnOnAssetsTTM")),
    }

# ---- Finnhub ----
def fetch_finnhub_snapshot(ticker: str) -> dict:
    if not FINNHUB_API_KEY:
        return {}
    base = "https://finnhub.io/api/v1"
    prof = http_get_json(
        url=f"{base}/stock/profile2",
        params={"symbol": ticker, "token": FINNHUB_API_KEY},
        timeout=HTTP_TIMEOUT,
        max_retries=MAX_RETRIES,
        sleep_sec=SLEEP_FINNHUB,
        cache_key=f"finnhub_profile_{ticker}",
    )
    basic = http_get_json(
        url=f"{base}/stock/metric",
        params={"symbol": ticker, "metric": "all", "token": FINNHUB_API_KEY},
        timeout=HTTP_TIMEOUT,
        max_retries=MAX_RETRIES,
        sleep_sec=SLEEP_FINNHUB,
        cache_key=f"finnhub_metric_{ticker}",
    )
    pt = http_get_json(
        url=f"{base}/stock/price-target",
        params={"symbol": ticker, "token": FINNHUB_API_KEY},
        timeout=HTTP_TIMEOUT,
        max_retries=MAX_RETRIES,
        sleep_sec=SLEEP_FINNHUB,
        cache_key=f"finnhub_pricetarget_{ticker}",
    )

    metric = (basic or {}).get("metric", {}) if isinstance(basic, dict) else {}
    return {
        "ticker": ticker,
        "market_cap": safe_num(prof.get("marketCapitalization")),
        "shares_outstanding": safe_num(prof.get("shareOutstanding")),
        "company_name": prof.get("name"),
        "sector": prof.get("finnhubIndustry"),
        "country": prof.get("country"),
        "beta": safe_num(metric.get("beta")),
        "pe_ttm": safe_num(metric.get("peTTM")),
        "ps_ttm": safe_num(metric.get("psTTM")),
        "pb_ttm": safe_num(metric.get("pbAnnual")),
        "52w_high": safe_num(metric.get("52WeekHigh")),
        "52w_low": safe_num(metric.get("52WeekLow")),
        "gross_margin_ttm": safe_num(metric.get("grossMarginTTM")),
        "operating_margin_ttm": safe_num(metric.get("operatingMarginTTM")),
        "net_margin_ttm": safe_num(metric.get("netMarginTTM")),
        "roe_ttm": safe_num(metric.get("roeTTM")),
        "roa_ttm": safe_num(metric.get("roaTTM")),
        # Analyst target
        "target_mean_price": safe_num(pt.get("targetMean")),
        "target_high_price": safe_num(pt.get("targetHigh")),
        "target_low_price": safe_num(pt.get("targetLow")),
    }

# ---- Alpha Vantage ----
def fetch_alphavantage_snapshot(ticker: str) -> dict:
    if not ALPHA_VANTAGE_API_KEY:
        return {}
    base = "https://www.alphavantage.co/query"
    ov = http_get_json(
        url=base,
        params={"function": "OVERVIEW", "symbol": ticker, "apikey": ALPHA_VANTAGE_API_KEY},
        timeout=HTTP_TIMEOUT,
        max_retries=MAX_RETRIES,
        sleep_sec=SLEEP_ALPHA,
        cache_key=f"av_overview_{ticker}",
    )
    # AlphaVantage returns dict with many string fields
    return {
        "ticker": ticker,
        "market_cap": safe_num(ov.get("MarketCapitalization")),
        "enterprise_value": safe_num(ov.get("EV")),
        "pe_ttm": safe_num(ov.get("PERatio")),
        "pe_forward": safe_num(ov.get("ForwardPE")),
        "peg_ratio": safe_num(ov.get("PEGRatio")),
        "ps_ttm": safe_num(ov.get("PriceToSalesRatioTTM")),
        "pb_ttm": safe_num(ov.get("PriceToBookRatio")),
        "ev_to_ebitda": safe_num(ov.get("EVToEBITDA")),
        "profit_margin": safe_num(ov.get("ProfitMargin")),
        "operating_margin": safe_num(ov.get("OperatingMarginTTM")),
        "roe_ttm": safe_num(ov.get("ReturnOnEquityTTM")),
        "roa_ttm": safe_num(ov.get("ReturnOnAssetsTTM")),
        "beta": safe_num(ov.get("Beta")),
        "52w_high": safe_num(ov.get("52WeekHigh")),
        "52w_low": safe_num(ov.get("52WeekLow")),
        "shares_outstanding": safe_num(ov.get("SharesOutstanding")),
    }

def fetch_snapshot_by_priority(ticker: str, priority: list) -> dict:
    for p in priority:
        try:
            if p == "fmp":
                d = fetch_fmp_snapshot(ticker)
            elif p == "finnhub":
                d = fetch_finnhub_snapshot(ticker)
            elif p == "alphavantage":
                d = fetch_alphavantage_snapshot(ticker)
            else:
                continue
            # accept if has at least one key besides ticker
            if d and any((k != "ticker" and v is not None) for k, v in d.items()):
                d["provider"] = p
                return d
        except Exception:
            continue
    return {"ticker": ticker, "provider": None}


In [ ]:
def fetch_external_fundamentals(tickers: list, max_tickers: int, priority: list) -> pd.DataFrame:
    tickers2 = [t for t in tickers if t != MARKET_TICKER]
    tickers2 = tickers2[:max_tickers]
    rows = []
    for i, t in enumerate(tickers2, 1):
        d = fetch_snapshot_by_priority(t, priority)
        rows.append(d)
        if i % 10 == 0:
            print(f"Fetched fundamentals: {i}/{len(tickers2)}")
    return pd.DataFrame(rows)

external_fundamentals = None
if ADD_EXTERNAL_FUNDAMENTALS:
    external_fundamentals = fetch_external_fundamentals(
        tickers=panel_f["ticker"].unique().tolist(),
        max_tickers=FUNDAMENTALS_MAX_TICKERS,
        priority=PROVIDER_PRIORITY
    )
    external_fundamentals.to_csv(OUTDIR / "external_fundamentals_snapshot.csv", index=False, encoding="utf-8")
    external_fundamentals.head()
else:
    print("External fundamentals disabled.")


## 11) Merge external fundamentals into dataset (ticker-level)
We merge by ticker to produce a unified EDA dataset.


In [ ]:
external_feature_cols = []
if external_fundamentals is not None and len(external_fundamentals):
    external_feature_cols = [c for c in external_fundamentals.columns if c not in ("ticker", "provider", "company_name", "sector", "industry", "country")]
    dataset = dataset.merge(external_fundamentals, on="ticker", how="left")
    print("Merged external fundamentals. New shape:", dataset.shape)
else:
    print("No external fundamentals merged.")


## 12) Train/Test split (70/30 by time)

In [ ]:
def time_based_split(df: pd.DataFrame, split_ratio: float = 0.7):
    if split_ratio <= 0 or split_ratio >= 1:
        raise ValueError("split_ratio must be between 0 and 1")
    dates = df["date"].dropna().sort_values().unique()
    if len(dates) < 2:
        raise ValueError("At least two distinct dates required")
    cut = int(len(dates) * split_ratio)
    # Ensure at least one train and one test row for two unique dates at ratio 0.1
    if len(dates) == 2 and split_ratio == 0.1:
        cut = 1
    cutoff_date = dates[cut - 1]
    train = df[df["date"] <= cutoff_date].copy()
    test = df[df["date"] > cutoff_date].copy()
    return train, test, cutoff_date

train, test, cutoff_date = time_based_split(dataset, 0.7)
cutoff_date, train.shape, test.shape


## 13) Feature Quality Report

In [ ]:
def feature_quality_report(df: pd.DataFrame, cols: list):
    rows = []
    for c in cols:
        s = df[c]
        rows.append({
            "column": c,
            "dtype": str(s.dtype),
            "missing_rate": float(s.isna().mean()),
            "mean": float(np.nanmean(s.values)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "std": float(np.nanstd(s.values)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "p01": float(np.nanpercentile(s.values, 1)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "p50": float(np.nanpercentile(s.values, 50)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "p99": float(np.nanpercentile(s.values, 99)) if pd.api.types.is_numeric_dtype(s) else np.nan,
        })
    return pd.DataFrame(rows)

all_feature_cols = trainable_feature_cols + external_feature_cols
quality = feature_quality_report(dataset, all_feature_cols + [target_dir_col, target_ret_col, target_max_up_col, target_max_dn_col])
quality.to_csv(OUTDIR / "feature_quality.csv", index=False, encoding="utf-8")
quality.sort_values("missing_rate", ascending=False).head(20)


## 14) Feature ranking (direction & return)

We compute rankings twice:
1) **Trainable-only features** (OHLCV-based) — safe for backtests  
2) **Trainable + external snapshot fundamentals** — for exploration & live scoring



In [ ]:
def rank_features(df: pd.DataFrame, feature_cols: list, target_dir_col: str, target_ret_col: str):
    sub = df[feature_cols + [target_dir_col, target_ret_col]].replace([np.inf, -np.inf], np.nan).dropna(subset=[target_dir_col, target_ret_col])
    y_dir = sub[target_dir_col].astype(int).values
    X = sub[feature_cols].fillna(0.0).values

    mi = mutual_info_classif(X, y_dir, discrete_features=False, random_state=RANDOM_SEED)

    corr_dir = []
    corr_ret = []
    for f in feature_cols:
        s = df[[f, target_dir_col]].dropna()
        corr_dir.append(float(s[f].corr(s[target_dir_col])) if len(s) > 1000 else np.nan)

        s2 = df[[f, target_ret_col]].dropna()
        corr_ret.append(float(s2[f].corr(s2[target_ret_col], method="spearman")) if len(s2) > 1000 else np.nan)

    rank_dir = pd.DataFrame({
        "feature": feature_cols,
        "mutual_info": mi,
        "corr_with_dir": corr_dir,
        "abs_corr_with_dir": np.abs(np.array(corr_dir, dtype=float)),
    }).sort_values(["mutual_info","abs_corr_with_dir"], ascending=False).reset_index(drop=True)

    rank_ret = pd.DataFrame({
        "feature": feature_cols,
        "spearman_corr_with_return": corr_ret,
        "abs_spearman_corr_with_return": np.abs(np.array(corr_ret, dtype=float)),
    }).sort_values(["abs_spearman_corr_with_return"], ascending=False).reset_index(drop=True)

    return rank_dir, rank_ret

rank_dir_train, rank_ret_train = rank_features(dataset, trainable_feature_cols, target_dir_col, target_ret_col)
rank_dir_train.to_csv(OUTDIR / "feature_ranking_direction_trainable.csv", index=False, encoding="utf-8")
rank_ret_train.to_csv(OUTDIR / "feature_ranking_return_trainable.csv", index=False, encoding="utf-8")

rank_dir_train.head(10), rank_ret_train.head(10)


In [ ]:
rank_dir_all, rank_ret_all = None, None
if external_feature_cols:
    all_cols = trainable_feature_cols + external_feature_cols
    rank_dir_all, rank_ret_all = rank_features(dataset, all_cols, target_dir_col, target_ret_col)
    rank_dir_all.to_csv(OUTDIR / "feature_ranking_direction_all.csv", index=False, encoding="utf-8")
    rank_ret_all.to_csv(OUTDIR / "feature_ranking_return_all.csv", index=False, encoding="utf-8")
    rank_dir_all.head(10), rank_ret_all.head(10)
else:
    print("No external fundamentals columns available (or not merged).")


## 15) Suggested Feature Set (MVP)
We pick top MI features and remove highly correlated ones.


In [ ]:
def suggest_feature_set(df: pd.DataFrame, rank_dir: pd.DataFrame, max_features=30, corr_threshold=0.95):
    candidates = rank_dir["feature"].tolist()[:max_features * 2]
    sub = df[candidates].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if len(sub) > SAMPLE_ROWS_FOR_CORR:
        sub = sub.sample(SAMPLE_ROWS_FOR_CORR, random_state=RANDOM_SEED)
    corr = sub.corr().abs()

    selected = []
    for f in candidates:
        if not selected:
            selected.append(f)
            continue
        if (corr.loc[f, selected] >= corr_threshold).any():
            continue
        selected.append(f)
        if len(selected) >= max_features:
            break
    return selected

suggested_trainable = suggest_feature_set(dataset, rank_dir_train, max_features=30, corr_threshold=0.95)
suggested_all = suggest_feature_set(dataset, rank_dir_all, max_features=30, corr_threshold=0.95) if rank_dir_all is not None else None

suggested_trainable, suggested_all


## 16) Save dataset (sample) + Suggested features JSON

In [ ]:
# Save suggested features
payload = {
    "horizon_days": HORIZON_DAYS,
    "cutoff_date_70_30": str(cutoff_date),
    "suggested_trainable_features": suggested_trainable,
    "suggested_all_features": suggested_all,
    "trainable_candidate_features": trainable_feature_cols,
    "external_fundamentals_features": external_feature_cols,
}
(OUTDIR / "suggested_features.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")

# Dataset sample
sample = dataset.dropna(subset=[target_ret_col]).copy()
if len(sample) > 200_000:
    sample = sample.sample(200_000, random_state=RANDOM_SEED)
sample.to_parquet(OUTDIR / "dataset_sample.parquet", index=False)

if SAVE_FULL_DATASET:
    dataset.to_parquet(OUTDIR / "dataset_full.parquet", index=False)

print("Saved parquet + JSON into:", OUTDIR.resolve())


## 17) Export ALL EDA outputs to Excel (multi-sheet)

The workbook includes:
- `Overview`
- `FeatureQuality`
- `Ranking_Direction_Trainable`
- `Ranking_Return_Trainable`
- `Ranking_Direction_All` (if available)
- `Ranking_Return_All` (if available)
- `SuggestedFeatures`
- `ExternalFundamentals` (if available)
- `DatasetSample` (capped to `MAX_ROWS_PER_SHEET`)


In [ ]:
def df_to_excel(writer, df: pd.DataFrame, sheet_name: str, max_rows: int):
    if df is None:
        return
    if len(df) > max_rows:
        df2 = df.head(max_rows).copy()
    else:
        df2 = df
    df2.to_excel(writer, sheet_name=sheet_name[:31], index=False)

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    df_to_excel(writer, overview, "Overview", MAX_ROWS_PER_SHEET)
    df_to_excel(writer, quality, "FeatureQuality", MAX_ROWS_PER_SHEET)
    df_to_excel(writer, rank_dir_train, "Ranking_Direction_Trainable", MAX_ROWS_PER_SHEET)
    df_to_excel(writer, rank_ret_train, "Ranking_Return_Trainable", MAX_ROWS_PER_SHEET)

    if rank_dir_all is not None:
        df_to_excel(writer, rank_dir_all, "Ranking_Direction_All", MAX_ROWS_PER_SHEET)
    if rank_ret_all is not None:
        df_to_excel(writer, rank_ret_all, "Ranking_Return_All", MAX_ROWS_PER_SHEET)

    # Suggested features as a small table
    sug_rows = []
    for f in suggested_trainable:
        sug_rows.append({"feature": f, "group": "trainable"})
    if suggested_all:
        for f in suggested_all:
            sug_rows.append({"feature": f, "group": "all_with_snapshot"})
    df_to_excel(writer, pd.DataFrame(sug_rows), "SuggestedFeatures", MAX_ROWS_PER_SHEET)

    if external_fundamentals is not None:
        df_to_excel(writer, external_fundamentals, "ExternalFundamentals", MAX_ROWS_PER_SHEET)

    df_to_excel(writer, sample, "DatasetSample", MAX_ROWS_PER_SHEET)

print("Excel exported to:", EXCEL_PATH.resolve())


## Done ✅

You now have:
- Trainable features (OHLCV-based) including **MACD**
- Optional external snapshot fundamentals from FMP/Finnhub/AlphaVantage
- All EDA outputs exported to **Excel** for visual inspection

### Next (Step 2)
- Train baseline models (logistic/ridge) → boosted trees (LightGBM/XGBoost)
- Evaluate:
  - Direction accuracy
  - Return error
  - Ride/simulation rules (TP/SL) for profitability
